## Автоэнкодеры и их типы (продолжение)

Загружаем данные и преобразовываем их в необходимый вид.

In [ ]:
from keras.datasets import mnist
import numpy as np

(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = x_train.astype('float32') / 255.
x_test  = x_test .astype('float32') / 255.
x_train = np.reshape(x_train, (len(x_train), 28, 28, 1))
x_test  = np.reshape(x_test,  (len(x_test),  28, 28, 1))

Функция отрисовки цифр

In [ ]:
%matplotlib inline
import seaborn as sns
import matplotlib.pyplot as plt

def plot_digits(*args):
    args = [x.squeeze() for x in args]
    n = min([x.shape[0] for x in args])
    
    plt.figure(figsize=(2*n, 2*len(args)))
    for j in range(n):
        for i in range(len(args)):
            ax = plt.subplot(len(args), n, i*n + j + 1)
            plt.imshow(args[i][j])
            plt.gray()
            ax.get_xaxis().set_visible(False)
            ax.get_yaxis().set_visible(False)

    plt.show()

Рассмотрим несколько видов автоэнкодеров. На прошлом занятии мы попробовали использовать сжимающий автоэнкодер.

## Deep autoencoder / Глубокий автоэнкодер

Глубокий автоэнкодер кроме входного, выходного и сжимающего внутреннего слоя состоит из нескольких полносвязных внутренних слоев. Вы можете добавлять/изменять их количество. Увеличивая число внутренних слоев, мы сможем находить больше нелинейных зависимостей в данных.

In [ ]:
from keras.layers import Input, Dense, Flatten, Reshape
from keras.models import Model

def create_deep_dense_ae():
    # Размерность кодированного представления
    encoding_dim = 49

    # Энкодер
    input_img = Input(shape=(28, 28, 1))
    flat_img = Flatten()(input_img)
    x = Dense(encoding_dim*3, activation='relu')(flat_img)
    x = Dense(encoding_dim*2, activation='relu')(x)
    encoded = Dense(encoding_dim, activation='linear')(x)
    
    # Декодер
    input_encoded = Input(shape=(encoding_dim,))
    x = Dense(encoding_dim*2, activation='relu')(input_encoded)
    x = Dense(encoding_dim*3, activation='relu')(x)
    flat_decoded = Dense(28*28, activation='sigmoid')(x)
    decoded = Reshape((28, 28, 1))(flat_decoded)
    
    # Модели
    encoder = Model(input_img, encoded, name="encoder")
    decoder = Model(input_encoded, decoded, name="decoder")
    autoencoder = Model(input_img, decoder(encoder(input_img)), name="autoencoder")
    return encoder, decoder, autoencoder

d_encoder, d_decoder, d_autoencoder = create_deep_dense_ae()
d_autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

Посмотрим число параметров нашей модели глубокого автоэнкодера

In [ ]:
d_encoder.summary()

In [ ]:
d_decoder.summary()

In [ ]:
d_autoencoder.summary()

In [ ]:
d_autoencoder.fit(x_train, x_train,
                  epochs=10,
                  batch_size=256,
                  shuffle=True,
                  validation_data=(x_test, x_test))

In [ ]:
vec49 = d_encoder.predict(x_test)

### Задание 1

Выполните предсказание для первых 10 изображений последовательно: энкодер потом декодер.

In [ ]:
n = 15

imgs = x_test[n:2*n]
# Ваш код

### Задание 2

Теперь предсказание выполните с использованием полной сборки модели автоэнкодера. Сравните результаты последовательного прогона и текущего. Что можете сказать по этому поводу?

In [ ]:
n = 10

imgs = x_test[:n]
# Ваш код

## Convolutional Autoencoder / Сверточный автоэнкодер

Так как мы работаем с картинками, в данных должна присутствовать некоторая пространственная инвариантность. Попробуем этим воспользоваться: построим сверточный автоэнкодер.

In [ ]:
from keras.layers import Conv2D, MaxPooling2D, UpSampling2D

def create_deep_conv_ae():
    input_img = Input(shape=(28, 28, 1))

    x = Conv2D(128, (7, 7), activation='relu', padding='same')(input_img)
    x = MaxPooling2D((2, 2), padding='same')(x)
    x = Conv2D(32, (2, 2), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2), padding='same')(x)
    encoded = Conv2D(1, (7, 7), activation='relu', padding='same')(x)

    # На этом моменте представление  (7, 7, 1) т.е. 49-размерное

    input_encoded = Input(shape=(7, 7, 1))
    x = Conv2D(32, (7, 7), activation='relu', padding='same')(input_encoded)
    x = UpSampling2D((2, 2))(x)
    x = Conv2D(128, (2, 2), activation='relu', padding='same')(x)
    x = UpSampling2D((2, 2))(x)
    decoded = Conv2D(1, (7, 7), activation='sigmoid', padding='same')(x)

    # Модели
    encoder = Model(input_img, encoded, name="encoder")
    decoder = Model(input_encoded, decoded, name="decoder")
    autoencoder = Model(input_img, decoder(encoder(input_img)), name="autoencoder")
    return encoder, decoder, autoencoder

c_encoder, c_decoder, c_autoencoder = create_deep_conv_ae()
c_autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

c_autoencoder.summary()

Параметр padding при создании слоя может принимать 3 значения: same, valid, causal. Этот параметр определяет как заполнить пустые ячейки данных, когда размерность на входе и на выходе различаются, то есть выходные данные большей размерности, чем входные. Данные нужно заполнить.

Если параметр - same, то все значения большей размерности удаляются, чтобы размерность выходных данных и входных были равны.

Если параметр - valid, то паддинг не используется.

 Если параметр - causal, то значения заполняются нулями.

In [ ]:
c_autoencoder.fit(x_train, x_train,
                epochs=4,
                batch_size=1024,
                shuffle=True,
                validation_data=(x_test, x_test))

### Задание 3

Выполните задание 1 для сверточного автоэнкодера.

In [ ]:
n = 10

imgs = x_test[:n]

# Ваш код

### Задание 4

Выполните задание 2 для сверточного автоэнкодера.

In [ ]:
n = 10

imgs = x_test[:n]

# Ваш код

### Задание 5

Сравните значения функции ошибки при обучении у двух предыдущих автоэнкодеров. Что можете сказать по этому поводу?